# Demo: traductor español ↔ quechua
NLLB-200-distilled-600M ajustado, un solo modelo para los dos sentidos. Funciona en Colab con GPU T4 (Runtime → Change runtime type → T4) o en CPU.

**Antes de ejecutar:** pon en `PESOS` el repo de Hugging Face (o la carpeta de Drive) con los pesos entrenados; el enlace está en el README.

El modelo se entrenó con quechua de Ayacucho. Con otras variedades (por ejemplo el quechua central) la calidad baja; ver el artículo.

In [ ]:
!pip -q install transformers sentencepiece sacrebleu pandas

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
PESOS = 'REEMPLAZAR/por-el-repo-de-pesos'   # p. ej. 'usuario/nllb600m-es-quy' o '/content/drive/MyDrive/mejor'
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(PESOS)
modelo = AutoModelForSeq2SeqLM.from_pretrained(PESOS).to(dev).eval()
COD = {'es': 'spa_Latn', 'qu': 'quy_Latn'}

def traducir(texto, origen='es', destino='qu'):
    # Mismos parámetros que el servicio: haz de 4 y sin repetir 3-gramas.
    tok.src_lang = COD[origen]
    enc = tok(texto, return_tensors='pt', truncation=True, max_length=128).to(dev)
    with torch.inference_mode():
        gen = modelo.generate(**enc, forced_bos_token_id=tok.convert_tokens_to_ids(COD[destino]),
                              num_beams=4, max_new_tokens=128, no_repeat_ngram_size=3)
    return tok.batch_decode(gen, skip_special_tokens=True)[0]


In [ ]:
# Español -> quechua
for f in ['Buenos días, ¿cómo estás?', 'La comunidad se reunirá mañana para sembrar papa.']:
    print(f, '->', traducir(f, 'es', 'qu'))
# Quechua -> español (el apóstrofo de las eyectivas puede escribirse con ' o ’; sin él algunas palabras cambian de sentido)
for q in ['Allin p’unchay', 'Allin punchay', 'Ch’aki allpa']:
    print(q, '->', traducir(q, 'qu', 'es'))


## Reproducir la evaluación en la prueba
Usa las 2 662 oraciones de la prueba sin texto religioso, que el repositorio guarda en `results/exp_norm15k_s0/test_predicciones.tsv` (columnas `src` y `ref`). Para correrla, clona el repositorio en Colab o sube ese archivo; usa 200 oraciones al azar para que dure pocos minutos.

In [ ]:
import pandas as pd
from sacrebleu.metrics import CHRF, BLEU
RUTA = 'results/exp_norm15k_s0/test_predicciones.tsv'   # ajusta la ruta si subiste el archivo a otro lugar
pred = pd.read_csv(RUTA, sep='	', keep_default_na=False)
for d, (o, t) in {'es->qu': ('es', 'qu'), 'qu->es': ('qu', 'es')}.items():
    muestra = pred[pred.direccion == d].sample(200, random_state=0)
    hip = [traducir(s, o, t) for s in muestra['src']]
    print(d, 'chrF++', round(CHRF(word_order=2).corpus_score(hip, [muestra['ref'].tolist()]).score, 1),
          '| BLEU', round(BLEU().corpus_score(hip, [muestra['ref'].tolist()]).score, 1))
